# WorldSim v3.1 — Data Reinforcement + Quad-Model Training (0.8B/2B/4B/9B)
Fix system prompts, reinforce Task F + personality pairs, train and compare 4 model sizes


## 1. Environment & Prerequisite Check


In [ ]:
from pathlib import Path
import sys, os, json, time, math, subprocess, shutil

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from scripts.prepare_dataset import _training_system_prompts, _row_to_training_example
from scripts.common import read_jsonl

prompts = _training_system_prompts(REPO_ROOT, {})
assert "L3_EN" in prompts, "L3_EN system prompt not loaded — run WS-LLM-V31-CODE-FIXES first"
print(f"L3_EN system prompt: {prompts['L3_EN'][:80]}...")

# Verify O~T support
sample_row = {
    "task": "O",
    "prompt": "[TASK] O",
    "output": '{"public_claim":"test","private_truth":"truth","deception_style":"omission","lie_degree":0.5,"detection_risk":0.2,"confidence":0.8}',
    "layer": "L3",
    "schema_version": 3,
}
converted = _row_to_training_example(sample_row, prompts)
assert converted["messages"][0]["content"] == prompts["L3_EN"]
print("Task O conversion works")

GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
LLAMA_SERVER = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-server"

MODELS_HF = {
    "0.8B": "Qwen/Qwen3.5-0.8B-Base",
    "2B": "Qwen/Qwen3.5-2B-Base",
    "4B": "Qwen/Qwen3.5-4B-Base",
    "9B": "Qwen/Qwen3.5-9B-Base",
}

v3_logic_path = REPO_ROOT / "data" / "validated" / "batch_v3_01_english_logic" / "passed.jsonl"
v3_new_path = REPO_ROOT / "data" / "validated" / "batch_v3_02_new_tasks" / "passed.jsonl"
v2_train = REPO_ROOT / "data" / "final" / "worldsim-v2-mix" / "train.jsonl"

print(f"v3 logic:    {sum(1 for _ in open(v3_logic_path))} rows" if v3_logic_path.exists() else "v3 logic missing")
print(f"v3 new:      {sum(1 for _ in open(v3_new_path))} rows" if v3_new_path.exists() else "v3 new missing")
print(f"v2 train:    {sum(1 for _ in open(v2_train))} rows" if v2_train.exists() else "v2 train missing")
print(f"Batch configs: F={(REPO_ROOT / 'config' / 'batches' / 'batch_v31_04_task_f_reinforce.yaml').exists()}, pairs={(REPO_ROOT / 'config' / 'batches' / 'batch_v31_05_personality_pairs.yaml').exists()}")
print("Environment ready")


## 2. Generate Reinforcement Data


In [ ]:
from scripts.generate_data import generate_dataset, load_batch_plan

BATCH_RESULTS = {}
for batch_name in ["batch_v31_04_task_f_reinforce", "batch_v31_05_personality_pairs"]:
    batch_plan = load_batch_plan(REPO_ROOT, batch_id=batch_name)
    target = sum((batch_plan.get("task_counts") or batch_plan.get("tasks", {})).values())
    print(f"\n{'=' * 50}")
    print(f"Generating: {batch_name} ({target} requests)")
    print(f"{'=' * 50}")
    started = time.perf_counter()
    result = generate_dataset(REPO_ROOT, batch_plan=batch_plan)
    elapsed = time.perf_counter() - started
    BATCH_RESULTS[batch_name] = {"result": result, "elapsed_min": round(elapsed / 60, 1)}
    print(f"Done in {elapsed / 60:.1f} min", flush=True)


## 3. Validate New Data


In [ ]:
from scripts.validate_data import validate_file

VALIDATION_RESULTS = {}
for batch_name in ["batch_v31_04_task_f_reinforce", "batch_v31_05_personality_pairs"]:
    raw_dir = REPO_ROOT / "data" / "raw" / batch_name
    if not raw_dir.exists():
        print(f"Skipping {batch_name}: no raw data")
        continue
    latest = max(raw_dir.glob("*.jsonl"), key=lambda p: p.stat().st_mtime, default=None)
    if latest is None:
        print(f"Skipping {batch_name}: no JSONL files")
        continue
    validated_dir = REPO_ROOT / "data" / "validated" / batch_name
    report = validate_file(input_path=latest, validated_dir=validated_dir, repo_root=REPO_ROOT)
    VALIDATION_RESULTS[batch_name] = report
    print(f"{batch_name}: passed={report.get('passed', 0)}, failed={report.get('failed', 0)}")


## 4. Merge + Reassemble v3.1 Dataset


In [ ]:
from collections import Counter
from scripts.assemble_v3_dataset import assemble_v3_dataset
from scripts.common import write_jsonl

for batch_name in ["batch_v31_04_task_f_reinforce", "batch_v31_05_personality_pairs"]:
    new_path = REPO_ROOT / "data" / "validated" / batch_name / "passed.jsonl"
    if not new_path.exists():
        continue
    new_rows = read_jsonl(new_path)
    if not new_rows:
        continue
    existing = read_jsonl(v3_logic_path)
    backup = v3_logic_path.with_suffix(f".jsonl.bak_before_{batch_name}")
    if not backup.exists():
        shutil.copy2(v3_logic_path, backup)
    merged = existing + new_rows
    write_jsonl(v3_logic_path, merged)
    print(f"Merged {len(new_rows)} rows from {batch_name} -> total {len(merged)}")

V31_DATASET_ID = "worldsim-v31-mix"
assembly_result = assemble_v3_dataset(REPO_ROOT, dataset_id=V31_DATASET_ID, dev_ratio=0.1, seed=42)

v31_final_dir = REPO_ROOT / "data" / "final" / V31_DATASET_ID
v31_train = read_jsonl(v31_final_dir / "train.jsonl")
v31_dev = read_jsonl(v31_final_dir / "dev.jsonl")

print(f"\n{'=' * 60}")
print("V3.1 DATASET ASSEMBLED")
print(f"{'=' * 60}")
print(f"Train: {len(v31_train)}")
print(f"Dev:   {len(v31_dev)}")
print(f"Total: {len(v31_train) + len(v31_dev)}")

task_counts = Counter(row.get("task", "?") for row in v31_train)
print("\nTask distribution (train):")
for task, count in sorted(task_counts.items()):
    print(f"  {task}: {count}")


## 5. Convert to Training Format (with L3_EN)


In [ ]:
from scripts.convert_mixed_final_to_training_format import convert_mixed_final_to_training_format
from scripts.curriculum_order_v3 import curriculum_order_v3

V31_TRAINING_DIR = REPO_ROOT / "data" / "training" / V31_DATASET_ID

conversion_result = convert_mixed_final_to_training_format(
    repo_root=REPO_ROOT,
    input_train=v31_final_dir / "train.jsonl",
    input_dev=v31_final_dir / "dev.jsonl",
    source_manifest=assembly_result.manifest_path,
    output_dir=V31_TRAINING_DIR,
    dataset_id=V31_DATASET_ID,
)

train_converted = conversion_result.train_output
dev_converted = conversion_result.dev_output
train_curriculum = V31_TRAINING_DIR / "train_curriculum.jsonl"
converted_rows = read_jsonl(train_converted)
ordered_rows = curriculum_order_v3(converted_rows, seed=42)
write_jsonl(train_curriculum, ordered_rows)

train_count = len(ordered_rows)
dev_count = conversion_result.dev_count
effective_batch = 1 * 8
max_steps = max(1, math.ceil(train_count / effective_batch)) * 3
l3_en_count = sum(1 for row in ordered_rows if row.get("messages", [{}])[0].get("content", "").startswith("You are WorldSim"))

print(f"Train: {train_count}, Dev: {dev_count}, Max steps: {max_steps}")
print(f"Rows using L3_EN system prompt: {l3_en_count}")
print("Conversion complete")


## 6. Train All 4 Models


In [ ]:
from datetime import UTC, datetime
from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

TRAIN_RESULTS = {}

for size_label, model_name in MODELS_HF.items():
    run_id = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
    output_dir = REPO_ROOT / "outputs" / "baseline" / f"worldsim-v31-{size_label.lower()}" / run_id

    print(f"\n{'=' * 60}")
    print(f"Training {size_label}: {model_name}")
    print(f"Output: {output_dir}")
    print(f"{'=' * 60}", flush=True)

    gradient_accumulation_steps = 16 if size_label == "9B" else 8
    per_device_train_batch_size = 1
    effective_batch = per_device_train_batch_size * gradient_accumulation_steps
    size_steps = max(1, math.ceil(train_count / effective_batch)) * 3

    cfg = SmokeRunConfig(
        run_mode="baseline",
        model_name=model_name,
        train_file=train_curriculum,
        dev_file=dev_converted,
        output_dir=output_dir,
        max_steps=size_steps,
        max_train_samples=0,
        max_eval_samples=0,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=1e-4,
        logging_steps=10,
        eval_steps=100,
        save_steps=100,
        save_total_limit=1,
        require_qlora=True,
        seed=42,
        dry_run=False,
    )

    started = time.perf_counter()
    try:
        result = run_baseline_or_raise(cfg)
        elapsed = time.perf_counter() - started
        TRAIN_RESULTS[size_label] = {
            "result": result,
            "elapsed_min": round(elapsed / 60, 1),
            "train_loss": result.train_loss,
            "eval_loss": result.eval_loss,
            "output_dir": str(result.output_dir),
            "max_steps": size_steps,
            "gradient_accumulation_steps": gradient_accumulation_steps,
        }
        print(f"\n{size_label} COMPLETE ({elapsed / 60:.1f} min)")
        print(f"train_loss: {result.train_loss}")
        print(f"eval_loss:  {result.eval_loss}", flush=True)
    except Exception as exc:
        elapsed = time.perf_counter() - started
        TRAIN_RESULTS[size_label] = {
            "error": str(exc),
            "elapsed_min": round(elapsed / 60, 1),
            "max_steps": size_steps,
            "gradient_accumulation_steps": gradient_accumulation_steps,
        }
        print(f"\n{size_label} FAILED ({elapsed / 60:.1f} min): {exc}", flush=True)

print(f"\n{'=' * 60}")
print("TRAINING SUMMARY")
print(f"{'=' * 60}")
for size, info in TRAIN_RESULTS.items():
    if "error" in info:
        print(f"{size}: FAILED {info['error'][:80]} ({info['elapsed_min']} min)")
    else:
        print(f"{size}: train_loss={info['train_loss']:.4f} eval_loss={info['eval_loss']:.4f} ({info['elapsed_min']} min)")


## 7. GGUF Conversion — All Fine-tuned Models


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def merge_and_convert_gguf(model_name, output_dir, gguf_name):
    output_dir = Path(output_dir)
    adapter_dir = output_dir / "adapter"
    merged_dir = output_dir / "merged_bf16"
    fp16_gguf = output_dir / f"{gguf_name}-f16.gguf"
    q4_gguf = output_dir / f"{gguf_name}-q4_k_m.gguf"

    if not (merged_dir / "config.json").exists():
        print("  Merging LoRA...", flush=True)
        base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
        merged_dir.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(str(merged_dir))
        tokenizer.save_pretrained(str(merged_dir))
        del merged, base
        torch.cuda.empty_cache()

    if not fp16_gguf.exists():
        print("  HF -> GGUF FP16...", flush=True)
        proc = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"],
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise RuntimeError(f"GGUF failed: {proc.stderr[-300:]}")

    if not q4_gguf.exists():
        print("  Quantize Q4_K_M...", flush=True)
        proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"], capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Quantize failed: {proc.stderr[-300:]}")

    return q4_gguf

GGUF_RESULTS = {}
for size_label in ["0.8B", "2B", "4B", "9B"]:
    info = TRAIN_RESULTS.get(size_label, {})
    if "error" in info or "result" not in info:
        print(f"Skipping {size_label}: training failed")
        continue

    model_name = MODELS_HF[size_label]
    gguf_name = f"worldsim-v31-qwen3.5-{size_label.lower()}"
    print(f"\n=== {size_label} GGUF ===", flush=True)
    try:
        q4_path = merge_and_convert_gguf(model_name, info["output_dir"], gguf_name)
        final_path = GGUF_DIR / f"{gguf_name}-q4_k_m.gguf"
        shutil.copy2(q4_path, final_path)
        GGUF_RESULTS[size_label] = {"path": final_path, "size_mb": round(final_path.stat().st_size / 1e6)}
        print(f"  {final_path.name} ({GGUF_RESULTS[size_label]['size_mb']} MB)", flush=True)
    except Exception as exc:
        GGUF_RESULTS[size_label] = {"error": str(exc)}
        print(f"  {size_label} GGUF failed: {exc}", flush=True)


## 8. Convert Base 4B/9B to GGUF (for comparison)


In [ ]:
def convert_base_to_gguf(model_name, output_gguf, short_name):
    output_gguf = Path(output_gguf)
    if output_gguf.exists():
        print(f"  {output_gguf.name} exists")
        return output_gguf

    work_dir = REPO_ROOT / "artifacts" / "base_models" / short_name
    hf_dir = work_dir / "hf"
    fp16_gguf = work_dir / f"{short_name}-f16.gguf"

    if not (hf_dir / "config.json").exists():
        print(f"  Downloading {model_name}...", flush=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        hf_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(hf_dir))
        tokenizer.save_pretrained(str(hf_dir))
        del model
        torch.cuda.empty_cache()

    if not fp16_gguf.exists():
        proc = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(hf_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"],
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise RuntimeError(f"GGUF failed: {proc.stderr[-300:]}")

    if not output_gguf.exists():
        proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(output_gguf), "Q4_K_M"], capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Quantize failed: {proc.stderr[-300:]}")
    print(f"  {output_gguf.name} ({output_gguf.stat().st_size / 1e6:.0f} MB)", flush=True)
    return output_gguf

BASE_GGUFS = {}
for short, hf_name, filename in [
    ("qwen3.5-0.8b-base", "Qwen/Qwen3.5-0.8B-Base", "qwen3.5-0.8b-base-q4_k_m.gguf"),
    ("qwen3.5-2b-base", "Qwen/Qwen3.5-2B-Base", "qwen3.5-2b-base-q4_k_m.gguf"),
    ("qwen3.5-4b-base", "Qwen/Qwen3.5-4B-Base", "qwen3.5-4b-base-q4_k_m.gguf"),
    ("qwen3.5-9b-base", "Qwen/Qwen3.5-9B-Base", "qwen3.5-9b-base-q4_k_m.gguf"),
]:
    path = GGUF_DIR / filename
    print(f"\n=== Base {short} ===")
    try:
        BASE_GGUFS[short] = convert_base_to_gguf(hf_name, path, short)
    except Exception as exc:
        print(f"  FAILED: {exc}")


## 9. Optional Extended Comparison (8 Models)


In [ ]:
OPTIONAL_MODEL_PATHS = {}
for short, path in BASE_GGUFS.items():
    OPTIONAL_MODEL_PATHS[f"base::{short}"] = path
for size_label, info in GGUF_RESULTS.items():
    if "path" in info:
        OPTIONAL_MODEL_PATHS[f"ft::{size_label}"] = info["path"]

print("Optional comparison inventory:")
for key, path in OPTIONAL_MODEL_PATHS.items():
    print(f"  {key}: {path}")
print("If time/GPU allows, reuse dgx_spark_extended_comparison.ipynb helpers here for an 8-model run.")


## 10. Grand Summary


In [ ]:
print("=" * 70)
print("WORLDSIM V3.1 — QUAD MODEL RESULTS")
print("=" * 70)

print("\n[Dataset]")
print(f"  v3.1 train: {train_count}, dev: {dev_count}")
print(f"  L3_EN rows: {l3_en_count}")

print("\n[Training]")
for size in ["0.8B", "2B", "4B", "9B"]:
    info = TRAIN_RESULTS.get(size, {})
    if "error" in info:
        print(f"  {size}: FAILED {info['error'][:60]}")
    elif "train_loss" in info:
        print(f"  {size}: loss={info['train_loss']:.4f}/{info['eval_loss']:.4f} ({info['elapsed_min']} min)")

print("\n[GGUF]")
for size in ["0.8B", "2B", "4B", "9B"]:
    info = GGUF_RESULTS.get(size, {})
    if "path" in info:
        print(f"  {size} FT: {info['path'].name} ({info['size_mb']} MB)")
    elif "error" in info:
        print(f"  {size} FT: FAILED")

print("\n[Base GGUF]")
for short, path in BASE_GGUFS.items():
    print(f"  {short}: {path.name}")
print("=" * 70)
